# CS-781 Phase 2 — Experiment 6: Knowledge Distillation (v3 IMPROVED)

**Teacher:** Phase-2 Exp5 Model 4 partial/layerwise NER model (frozen).

**Student:** original Phase-1 Model 2 decoder-only LM + BiGRU NER head.

Student learns from both **gold BILOU labels** and **soft teacher logits**.

### v3 improvements over v2
- **Critical fix:** restores the missing `count_training_tags()` / `CLASS_WEIGHTS` construction (v2 would fail at the first hard-loss call).
- **Teacher-logit caching:** Exp5 teacher runs once per training window, not once every epoch; this makes KD much faster and frees GPU memory before student training.
- **Exact teacher reconstruction check:** evaluates with the checkpoint's own saved thresholds and compares against its stored Fold-2 F1.
- **Class-balanced KD:** the KL term is token-weighted with the same clipped class weights as the hard NER loss, preventing the huge `O` class from dominating distillation.
- **Layerwise student LR decay:** lower Phase-1 Model-2 blocks move more conservatively while upper blocks adapt more strongly.
- Keeps v2's packed BiGRU, FP32 stability, more optimizer updates, final-refit teacher handling, and dual heldout/refit submissions.


In [1]:
import os, sys, json, math, time, random, re, zipfile, subprocess, gc, hashlib
from pathlib import Path
from collections import Counter, defaultdict

try:
    import huggingface_hub
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download

SEED=781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
print("PyTorch:",torch.__version__)
print("Device:",DEVICE)
if DEVICE=="cuda": print("GPU:",torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [3]:
ROOT=Path("/kaggle/working/phase2_exp6_kd_m4teacher_m2student")
ROOT.mkdir(parents=True,exist_ok=True)
HF_REPO="Exploration-Lab/CS-781-Capstone"

# Attach these on Kaggle or set paths manually.
MODEL4_PHASE1_PATH=None
TEACHER_HELDOUT_PATH=None       # Exp5 best_partial_ner.pt (Fold1-trained / Fold2-selected)
TEACHER_REFIT_PATH=None         # optional: Exp5 final_refit_partial_ner.pt for final all-data KD
MODEL2_PHASE1_PATH=None
TOKENIZER_PATH=None

NER_SEQ_LEN=768
WINDOW_OVERLAP=192
WINDOW_STEP=NER_SEQ_LEN-WINDOW_OVERLAP

# Teacher + full student are both resident, so keep micro-batch conservative on T4.
TRAIN_BATCH_SIZE=1
EVAL_BATCH_SIZE=2
GRAD_ACCUM_STEPS=2
EPOCHS=10

# ---------- KD ----------
KD_TEMPERATURE=2.5
HARD_LOSS_WEIGHT=0.50
KD_LOSS_WEIGHT=0.50
KD_WEIGHT_BY_CLASS=True
CACHE_TEACHER_LOGITS=True
TEACHER_CACHE_DTYPE="float16"   # CPU cache only; KL is computed in float32
assert abs(HARD_LOSS_WEIGHT+KD_LOSS_WEIGHT-1.0)<1e-8

# Student optimization
STUDENT_BACKBONE_LR=5e-5
STUDENT_LAYERWISE_LR_DECAY=0.92
STUDENT_HEAD_LR=1e-3
HEAD_TYPE="bigru"
HEAD_HIDDEN=256
HEAD_DROPOUT=0.10

WEIGHT_DECAY=0.01
WARMUP_RATIO=0.08
MAX_GRAD_NORM=1.0
USE_CLASS_WEIGHTS=True

# Full FP32 is the safe default after the mixed-precision issues seen in prior experiments.
AMP=False

THRESHOLD_GRID=[round(x,2) for x in np.arange(0.00,0.81,0.05)]
DO_FINAL_REFIT_ON_ALL=True
USE_VITERBI=True
PRESPLIT_PUNCT=True
USE_PER_LABEL_THRESHOLDS=True
EARLY_STOP_PATIENCE=4
CHECK_TEACHER_F1=True   # score reconstructed teacher on Fold 2 before KD (sanity)

EXPECTED_TOKENIZER_SHA256="9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

BEST_KD_PATH=ROOT/"best_kd_student.pt"
FINAL_REFIT_PATH=ROOT/"final_refit_kd_student.pt"
VAL_PRED_PATH=ROOT/"val_prediction.json"
TEST_PRED_PATH=ROOT/"prediction.json"
SUBMISSION_ZIP=ROOT/"submission.zip"

print("ROOT:",ROOT)
print("KD temperature:",KD_TEMPERATURE)
print("Hard/KD weights:",HARD_LOSS_WEIGHT,KD_LOSS_WEIGHT)
print("Student top-block/head LR:",STUDENT_BACKBONE_LR,STUDENT_HEAD_LR)
print("Student layerwise LR decay:",STUDENT_LAYERWISE_LR_DECAY)
print("Teacher cache:",CACHE_TEACHER_LOGITS,TEACHER_CACHE_DTYPE,"class-balanced KD:",KD_WEIGHT_BY_CLASS)
print("Student head:",HEAD_TYPE,"hidden:",HEAD_HIDDEN)
print("AMP:",AMP)


ROOT: /kaggle/working/phase2_exp6_kd_m4teacher_m2student
KD temperature: 2.5
Hard/KD weights: 0.5 0.5
Student top-block/head LR: 5e-05 0.001
Student layerwise LR decay: 0.92
Teacher cache: True float16 class-balanced KD: True
Student head: bigru hidden: 256
AMP: False


In [4]:
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def discover_inputs():
    m4=[]; th=[]; tr=[]; m2=[]; toks=[]
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.exists(base): continue
        for root,dirs,files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn); low=fn.lower()
                if low.endswith(".pt"):
                    if "eff_full_best" in low: m4.append(p)
                    elif "best_partial_ner" in low: th.append(p)
                    elif "final_refit_partial_ner" in low: tr.append(p)
                    elif "dec_full_best" in low or ("dec_full_latest" in low and "(8)" in low): m2.append(p)
                if low=="tokenizer.json" or ("tokenizer" in low and low.endswith(".json")):
                    toks.append(p)
    return sorted(m4),sorted(th),sorted(tr),sorted(m2),sorted(toks)

m4s,ths,trs,m2s,toks=discover_inputs()
print("Model4 Phase1 candidates:",*m4s,sep="\n  ")
print("Teacher heldout candidates:",*ths,sep="\n  ")
print("Teacher refit candidates:",*trs,sep="\n  ")
print("Model2 Phase1 candidates:",*m2s,sep="\n  ")

if MODEL4_PHASE1_PATH is None:
    exact=[p for p in m4s if "eff_full_best (6).pt" in os.path.basename(p).lower()]
    MODEL4_PHASE1_PATH=exact[-1] if exact else (m4s[-1] if m4s else None)
if TEACHER_HELDOUT_PATH is None:
    TEACHER_HELDOUT_PATH=ths[-1] if ths else None
if TEACHER_REFIT_PATH is None and trs:
    TEACHER_REFIT_PATH=trs[-1]
if MODEL2_PHASE1_PATH is None:
    MODEL2_PHASE1_PATH=m2s[-1] if m2s else None
if TOKENIZER_PATH is None:
    exact=[p for p in toks if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
    TOKENIZER_PATH=exact[-1] if exact else None

assert MODEL4_PHASE1_PATH and os.path.exists(MODEL4_PHASE1_PATH)
assert TEACHER_HELDOUT_PATH and os.path.exists(TEACHER_HELDOUT_PATH), (
    "Attach Exp5 best_partial_ner.pt — this is the held-out teacher used during Fold1->Fold2 KD selection."
)
assert MODEL2_PHASE1_PATH and os.path.exists(MODEL2_PHASE1_PATH)
assert TOKENIZER_PATH and os.path.exists(TOKENIZER_PATH)
assert sha256_file(TOKENIZER_PATH)==EXPECTED_TOKENIZER_SHA256

print("\nUSING")
print(" Model4 Phase1:",MODEL4_PHASE1_PATH)
print(" Teacher heldout:",TEACHER_HELDOUT_PATH)
print(" Teacher refit:",TEACHER_REFIT_PATH if TEACHER_REFIT_PATH else "(not attached; heldout teacher will be reused for final KD)")
print(" Model2 Phase1:",MODEL2_PHASE1_PATH)
print(" Tokenizer:",TOKENIZER_PATH)
print(" Tokenizer SHA:",sha256_file(TOKENIZER_PATH))


Model4 Phase1 candidates:
  /kaggle/input/datasets/aaryant23/phase2-kd/eff_full_best (6).pt
Teacher heldout candidates:
  /kaggle/input/datasets/aaryant23/phase2-kd/best_partial_ner.pt
Teacher refit candidates:
  /kaggle/input/datasets/aaryant23/phase2-kd/final_refit_partial_ner.pt
Model2 Phase1 candidates:
  /kaggle/input/datasets/aaryant23/phase2-kd/dec_full_best (8).pt

USING
 Model4 Phase1: /kaggle/input/datasets/aaryant23/phase2-kd/eff_full_best (6).pt
 Teacher heldout: /kaggle/input/datasets/aaryant23/phase2-kd/best_partial_ner.pt
 Teacher refit: /kaggle/input/datasets/aaryant23/phase2-kd/final_refit_partial_ner.pt
 Model2 Phase1: /kaggle/input/datasets/aaryant23/phase2-kd/dec_full_best (8).pt
 Tokenizer: /kaggle/input/datasets/aaryant23/phase2-kd/tokenizer.json
 Tokenizer SHA: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [5]:
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]

class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]; self.itos=list(BPE_SPECIALS)
        self.stoi={t:i for i,t in enumerate(self.itos)}; self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word); ranks=self._ranks
        while len(symbols)>1:
            best=None; best_rank=None
            for j in range(len(symbols)-1):
                r=ranks.get((symbols[j],symbols[j+1]))
                if r is not None and (best_rank is None or r<best_rank):
                    best=(symbols[j],symbols[j+1]); best_rank=r
            if best is None: break
            left,right=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==left and symbols[j+1]==right:
                    merged.append(left+right); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",text):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok in BPE_SPECIALS: continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        obj=json.load(open(path,encoding="utf-8")); tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]; tok.itos=list(obj["itos"])
        tok.stoi={t:i for i,t in enumerate(tok.itos)}; tok.vocab_size=len(tok.itos)
        tok.symbol_bytes={}
        for b in range(256): tok.symbol_bytes[f"{b:02x}"]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_PATH)
print("Tokenizer vocab:",len(tokenizer))

UNIT_RE=re.compile(r"\w+|[^\w\s]|\s+" if PRESPLIT_PUNCT else r"\S+|\s+")
print("Token unit regex:",UNIT_RE.pattern)

def encode_with_char_offsets(text):
    ids=[]; offsets=[]; exact=[]; unk=tokenizer.unk_token_id
    for m in UNIT_RE.finditer(text):
        unit=m.group(0); base=m.start(); symbols=tokenizer.encode_word(unit)
        byte_to_char={0:0}; bpos=0
        for ci,ch in enumerate(unit):
            bpos+=len(ch.encode("utf-8")); byte_to_char[bpos]=ci+1
        boundaries=sorted(byte_to_char)
        sb=0
        for sym in symbols:
            raw=tokenizer.symbol_bytes.get(sym)
            if raw is None:
                ids.append(unk); offsets.append((m.start(),m.end())); exact.append(False); continue
            eb=sb+len(raw); ok=(sb in byte_to_char and eb in byte_to_char)
            if ok:
                cs=base+byte_to_char[sb]; ce=base+byte_to_char[eb]
            else:
                lb=max(x for x in boundaries if x<=sb); rb=min(x for x in boundaries if x>=eb)
                cs=base+byte_to_char[lb]; ce=base+byte_to_char[rb]
            ids.append(tokenizer.stoi.get(sym,unk)); offsets.append((cs,ce)); exact.append(ok); sb=eb
    return ids,offsets,exact

probe="IN THE SUPREME COURT OF INDIA\\nSection 14: ₹100 — legal test."
pids,poff,pexact=encode_with_char_offsets(probe)
assert tokenizer.decode(pids)==probe
print("Round-trip exact: True")
print("Probe exact-boundary tokens:",sum(pexact),"/",len(pexact))


Tokenizer vocab: 24576
Token unit regex: \w+|[^\w\s]|\s+
Round-trip exact: True
Probe exact-boundary tokens: 27 / 27


In [6]:
PHASE2_FILES={
    "fold1_text":"Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold":"Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text":"Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold":"Phase_2_LNER/Train/fold2-gold.json",
    "labels":"Phase_2_LNER/Train/labels.txt",
    "test":"Phase_2_LNER/Test/fold3-text.json",
}
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        print("✅ Using HF_TOKEN from Kaggle Secrets (token value is not printed).")
except Exception:
    print("ℹ️ HF_TOKEN secret not available; trying public Hugging Face access.")

def get_hf_file(filename):
    return hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=filename,
        token=HF_TOKEN if HF_TOKEN else None,
    )
phase2_paths={k:get_hf_file(v) for k,v in PHASE2_FILES.items()}
for k,v in phase2_paths.items(): print(f"{k:12s} -> {v}")

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

with open(phase2_paths["labels"],encoding="utf-8") as f:
    LABELS=[x.strip() for x in f if x.strip()]
assert len(LABELS)==12 and len(set(LABELS))==12,LABELS
print("Official labels:",LABELS)


✅ Using HF_TOKEN from Kaggle Secrets (token value is not printed).


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

fold1_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json
fold1_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json
fold2_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json
fold2_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json
labels       -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt
test         -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb

In [7]:
def normalize_text_json(obj):
    if isinstance(obj,dict):
        if all(isinstance(v,str) for v in obj.values()): return {str(k):v for k,v in obj.items()}
        if "data" in obj: return normalize_text_json(obj["data"])
    if isinstance(obj,list):
        out={}
        for r in obj:
            did=r.get("id",r.get("doc_id",r.get("document_id")))
            txt=r.get("text",r.get("document"))
            if did is None or not isinstance(txt,str): raise ValueError(f"Cannot parse text record: {r}")
            out[str(did)]=txt
        return out
    raise TypeError(f"Unsupported text JSON type: {type(obj)}")

def normalize_one_entity(e):
    if isinstance(e,list):
        if len(e) not in (3,4): raise ValueError(f"Bad entity: {e}")
        a,b,l=e[:3]; et=e[3] if len(e)==4 else None
        return int(a),int(b),str(l),et
    if isinstance(e,dict):
        a=e.get("start",e.get("start_char",e.get("start_character")))
        b=e.get("end",e.get("end_char",e.get("end_character")))
        l=e.get("label",e.get("type",e.get("entity_type")))
        et=e.get("text",e.get("entity_text"))
        if a is None or b is None or l is None: raise ValueError(f"Cannot parse entity: {e}")
        return int(a),int(b),str(l),et
    raise TypeError(type(e))

def normalize_gold_json(obj):
    if isinstance(obj,dict) and "data" in obj: return normalize_gold_json(obj["data"])
    if isinstance(obj,dict):
        out={}
        for did,ents in obj.items():
            if isinstance(ents,dict): ents=ents.get("entities",ents.get("annotations",ents.get("spans")))
            if ents is None: ents=[]
            out[str(did)]=[normalize_one_entity(e) for e in ents]
        return out
    if isinstance(obj,list):
        out={}
        for r in obj:
            did=r.get("id",r.get("doc_id",r.get("document_id")))
            ents=r.get("entities",r.get("annotations",r.get("spans")))
            if did is None or ents is None: raise ValueError(f"Cannot parse gold record: {r}")
            out[str(did)]=[normalize_one_entity(e) for e in ents]
        return out
    raise TypeError(type(obj))

fold1_text=normalize_text_json(read_json(phase2_paths["fold1_text"]))
fold2_text=normalize_text_json(read_json(phase2_paths["fold2_text"]))
test_text=normalize_text_json(read_json(phase2_paths["test"]))
fold1_gold=normalize_gold_json(read_json(phase2_paths["fold1_gold"]))
fold2_gold=normalize_gold_json(read_json(phase2_paths["fold2_gold"]))
assert set(fold1_text)==set(fold1_gold)
assert set(fold2_text)==set(fold2_gold)

allowed=set(LABELS)
def validate_gold(text_map,gold_map,name):
    n_ent=0; bad_text=0; overlaps=0; counts=Counter(); lengths=[]
    for did,text in text_map.items():
        prev=[]
        for a,b,l,et in gold_map[did]:
            assert 0<=a<b<=len(text),(did,a,b,len(text))
            assert l in allowed,(did,l)
            if et is not None and text[a:b]!=et: bad_text+=1
            for aa,bb,ll,_ in prev:
                if max(a,aa)<min(b,bb): overlaps+=1
            prev.append((a,b,l,et)); n_ent+=1; counts[l]+=1; lengths.append(b-a)
    print(f"{name}: docs={len(text_map):,} entities={n_ent:,} entity_text_mismatch={bad_text} overlapping_pairs={overlaps}")
    print(" label counts:",dict(counts))
    if lengths:
        print(" entity char length p50/p90/max:",int(np.percentile(lengths,50)),int(np.percentile(lengths,90)),max(lengths))

validate_gold(fold1_text,fold1_gold,"Fold1")
validate_gold(fold2_text,fold2_gold,"Fold2")
print("Test docs:",len(test_text))


Fold1: docs=35 entities=4,120 entity_text_mismatch=0 overlapping_pairs=0
 label counts: {'RESP': 191, 'DATE': 813, 'COURT': 455, 'WIT': 67, 'APP': 207, 'STAT': 525, 'A.COUNSEL': 70, 'PREC': 624, 'JUDGE': 123, 'R.COUNSEL': 87, 'CASENO': 423, 'AUTH': 535}
 entity char length p50/p90/max: 13 34 195
Fold2: docs=35 entities=4,273 entity_text_mismatch=0 overlapping_pairs=0
 label counts: {'JUDGE': 122, 'CASENO': 337, 'APP': 200, 'RESP': 190, 'A.COUNSEL': 125, 'R.COUNSEL': 97, 'STAT': 735, 'DATE': 762, 'AUTH': 468, 'PREC': 559, 'COURT': 558, 'WIT': 120}
 entity char length p50/p90/max: 13 32 233
Test docs: 35


In [8]:
TAG_NAMES=["O"]
for lab in LABELS: TAG_NAMES += [f"B-{lab}",f"I-{lab}",f"L-{lab}",f"U-{lab}"]
TAG2ID={t:i for i,t in enumerate(TAG_NAMES)}
ID2TAG={i:t for t,i in TAG2ID.items()}
O_ID=TAG2ID["O"]; IGNORE_INDEX=-100; N_TAGS=len(TAG_NAMES)

def tag_document(text,entities):
    ids,offsets,exact=encode_with_char_offsets(text)
    tags=[O_ID]*len(ids); group=[-1]*len(ids)
    start_to_idx={}; end_to_idx={}
    for i,((a,b),ex) in enumerate(zip(offsets,exact)):
        if ex:
            start_to_idx.setdefault(a,i); end_to_idx[b]=i
    stats=Counter(); ranges=[]
    for gi,(a,b,lab,et) in enumerate(sorted(entities,key=lambda z:(z[0],z[1],z[2]))):
        if a not in start_to_idx or b not in end_to_idx:
            stats["unaligned"]+=1; continue
        i=start_to_idx[a]; j=end_to_idx[b]
        if j<i or offsets[i][0]!=a or offsets[j][1]!=b:
            stats["unaligned"]+=1; continue
        if not all(offsets[k][1]==offsets[k+1][0] for k in range(i,j)):
            stats["unaligned"]+=1; continue
        if any(group[k]!=-1 for k in range(i,j+1)):
            stats["overlap_conflict"]+=1; continue
        if i==j:
            tags[i]=TAG2ID[f"U-{lab}"]
        else:
            tags[i]=TAG2ID[f"B-{lab}"]
            for k in range(i+1,j): tags[k]=TAG2ID[f"I-{lab}"]
            tags[j]=TAG2ID[f"L-{lab}"]
        for k in range(i,j+1): group[k]=gi
        ranges.append((i,j,gi)); stats["aligned"]+=1
    return ids,offsets,tags,group,ranges,stats

def alignment_report(text_map,gold_map,name):
    total=Counter(); tok_lens=[]
    for did,text in text_map.items():
        ids,offsets,tags,group,ranges,st=tag_document(text,gold_map[did])
        total.update(st); tok_lens.append(len(ids))
    denom=total["aligned"]+total["unaligned"]+total["overlap_conflict"]
    print(f"{name} alignment:",dict(total))
    print("Representable gold spans:",f"{total['aligned']}/{denom} = {total['aligned']/max(1,denom):.4%}")
    print("Token lengths p50/p90/p95/max:",[int(np.percentile(tok_lens,q)) for q in (50,90,95)],max(tok_lens))

alignment_report(fold1_text,fold1_gold,"Fold1")
alignment_report(fold2_text,fold2_gold,"Fold2")


Fold1 alignment: {'aligned': 4093, 'unaligned': 27}
Representable gold spans: 4093/4120 = 99.3447%
Token lengths p50/p90/p95/max: [10181, 19167, 21017] 40485
Fold2 alignment: {'aligned': 4257, 'unaligned': 16}
Representable gold spans: 4257/4273 = 99.6256%
Token lengths p50/p90/p95/max: [10734, 16185, 24784] 44587


In [9]:
import bisect

class NERWindowDataset(Dataset):
    def __init__(self,text_map,gold_map=None,seq_len=NER_SEQ_LEN,step=WINDOW_STEP):
        self.text_map=text_map; self.gold_map=gold_map; self.seq_len=seq_len; self.step=step
        self.docs={}; self.windows=[]
        t0=time.time()
        for di,(did,text) in enumerate(text_map.items()):
            if gold_map is None:
                ids,offsets,exact=encode_with_char_offsets(text)
                tags=None; group=None; ranges=[]; stats=Counter()
            else:
                ids,offsets,tags,group,ranges,stats=tag_document(text,gold_map[did])
            ranges=sorted(ranges)                                   # (first_tok, last_tok, gold_idx), non-overlapping
            self.docs[did]=dict(text=text,ids=ids,offsets=offsets,tags=tags,group=group,ranges=ranges,
                                r_last=[r[1] for r in ranges],stats=stats)
            n=len(ids)
            if n==0: self.windows.append((did,0,0))
            else:
                for s in range(0,max(1,n),step):
                    e=min(n,s+seq_len); self.windows.append((did,s,e))
                    if e==n: break
            if (di+1)%100==0: print(f" tokenized {di+1}/{len(text_map)} docs",flush=True)
        print(f"Built {len(self.windows):,} windows from {len(self.docs):,} docs in {(time.time()-t0)/60:.1f} min")
    def __len__(self): return len(self.windows)
    def __getitem__(self,idx):
        did,s,e=self.windows[idx]; d=self.docs[did]; ids=d["ids"][s:e]
        item={"doc_id":did,"start_token":s,"end_token":e,"input_ids":torch.tensor(ids,dtype=torch.long)}
        if d["tags"] is not None:
            labels=list(d["tags"][s:e])
            # entities that are cut by the window border are ignored (no loss on their visible fragment)
            k=bisect.bisect_left(d["r_last"],s)
            while k<len(d["ranges"]):
                i,j,_=d["ranges"][k]
                if i>=e: break
                if i<s or j>=e:
                    for p in range(max(i,s),min(j,e-1)+1): labels[p-s]=IGNORE_INDEX
                k+=1
            item["labels"]=torch.tensor(labels,dtype=torch.long)
        return item

def collate_windows(batch):
    B=len(batch); T=max(max(1,len(x["input_ids"])) for x in batch)
    input_ids=torch.full((B,T),tokenizer.pad_token_id,dtype=torch.long)
    attention_mask=torch.zeros((B,T),dtype=torch.bool)
    labels=torch.full((B,T),IGNORE_INDEX,dtype=torch.long) if "labels" in batch[0] else None
    doc_ids=[]; starts=[]; ends=[]
    for i,x in enumerate(batch):
        n=len(x["input_ids"])
        if n:
            input_ids[i,:n]=x["input_ids"]; attention_mask[i,:n]=True
            if labels is not None: labels[i,:n]=x["labels"]
        doc_ids.append(x["doc_id"]); starts.append(x["start_token"]); ends.append(x["end_token"])
    out={"input_ids":input_ids,"attention_mask":attention_mask,"doc_ids":doc_ids,"start_tokens":starts,"end_tokens":ends}
    if labels is not None: out["labels"]=labels
    return out

train_ds=NERWindowDataset(fold1_text,fold1_gold)
val_ds=NERWindowDataset(fold2_text,fold2_gold)
print("Train windows:",len(train_ds))
print("Val windows:",len(val_ds))


Built 736 windows from 35 docs in 0.0 min
Built 748 windows from 35 docs in 0.0 min
Train windows: 736
Val windows: 748


In [11]:
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-5):
        super().__init__(); self.weight=nn.Parameter(torch.ones(dim)); self.eps=eps
    def forward(self,x):
        dtype=x.dtype; x32=x.float()
        x32=x32*torch.rsqrt(x32.pow(2).mean(-1,keepdim=True)+self.eps)
        return x32.to(dtype)*self.weight

class RotaryTable(nn.Module):
    def __init__(self,head_dim,max_positions,base=10000.0):
        super().__init__(); assert head_dim%2==0
        inv=1.0/(base**(torch.arange(0,head_dim,2,dtype=torch.float32)/head_dim))
        freqs=torch.outer(torch.arange(max_positions,dtype=torch.float32),inv)
        self.register_buffer("cos_cached",freqs.cos(),persistent=False)
        self.register_buffer("sin_cached",freqs.sin(),persistent=False)
        self.max_positions=max_positions
    def forward(self,position_ids):
        return self.cos_cached[position_ids].unsqueeze(1),self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x,cos,sin):
    B,H,T,D=x.shape; cos=cos.to(x.dtype); sin=sin.to(x.dtype)
    xp=x.view(B,H,T,D//2,2); x0,x1=xp[...,0],xp[...,1]
    return torch.stack((x0*cos-x1*sin,x0*sin+x1*cos),dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    GQA_DECODE_PATH=True
    BIDIRECTIONAL=False   # set True for NER (no causal mask; padding mask only)
    def __init__(self,d_model,n_heads,n_kv_heads,dropout):
        super().__init__(); assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads; self.n_kv_heads=n_kv_heads; self.head_dim=d_model//n_heads
        self.kv_repeat=n_heads//n_kv_heads; self.dropout=dropout
        self.bidir_override=None   # per-block override of the class-level BIDIRECTIONAL switch
        self.q_proj=nn.Linear(d_model,n_heads*self.head_dim,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.o_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x,cos,sin,past_kv=None,use_cache=False,key_valid=None):
        B,T,C=x.shape; H,KV,D=self.n_heads,self.n_kv_heads,self.head_dim
        q=self.q_proj(x).view(B,T,H,D).transpose(1,2)
        k=self.k_proj(x).view(B,T,KV,D).transpose(1,2)
        v=self.v_proj(x).view(B,T,KV,D).transpose(1,2)
        q=apply_rope(q,cos,sin); k=apply_rope(k,cos,sin)
        past_len=0
        if past_kv is not None:
            pk,pv=past_kv; past_len=pk.size(2); k=torch.cat([pk,k],2); v=torch.cat([pv,v],2)
        new_kv=(k,v) if use_cache else None; S=k.size(2); drop=self.dropout if self.training else 0.0
        if T==1 and past_len>0 and self.GQA_DECODE_PATH:
            qg=q.reshape(B,KV,self.kv_repeat,D); mask=None if key_valid is None else key_valid[:,None,None,:]
            y=F.scaled_dot_product_attention(qg,k,v,attn_mask=mask,dropout_p=drop,is_causal=False)
            y=y.reshape(B,H,1,D).transpose(1,2).contiguous().view(B,1,C)
            return self.o_proj(y),new_kv
        if self.kv_repeat!=1:
            k_attn=k.repeat_interleave(self.kv_repeat,1); v_attn=v.repeat_interleave(self.kv_repeat,1)
        else: k_attn,v_attn=k,v
        attn_mask,is_causal=None,False
        _bidir=self.BIDIRECTIONAL if self.bidir_override is None else self.bidir_override
        if _bidir:
            if key_valid is not None: attn_mask=key_valid[:,None,None,:]
        elif T==1:
            if key_valid is not None: attn_mask=key_valid[:,None,None,:]
        elif key_valid is None and past_len==0:
            is_causal=True
        else:
            qi=torch.arange(T,device=x.device)[:,None]+past_len
            kj=torch.arange(S,device=x.device)[None,:]
            attn_mask=(kj<=qi)[None,None,:,:]
            if key_valid is not None:
                attn_mask=attn_mask & key_valid[:,None,None,:]
                if past_len==0: attn_mask=attn_mask | (kj==qi)[None,None,:,:]
        y=F.scaled_dot_product_attention(q,k_attn,v_attn,attn_mask=attn_mask,dropout_p=drop,is_causal=is_causal)
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.o_proj(y),new_kv

class SwiGLU(nn.Module):
    def __init__(self,d_model,hidden_dim,dropout):
        super().__init__(); self.gate_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.up_proj=nn.Linear(d_model,hidden_dim,bias=False); self.down_proj=nn.Linear(hidden_dim,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x): return self.dropout(self.down_proj(F.silu(self.gate_proj(x))*self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,n_kv_heads,ffn_dim,dropout,rms_eps):
        super().__init__(); self.norm1=RMSNorm(d_model,rms_eps)
        self.attn=GQACausalSelfAttention(d_model,n_heads,n_kv_heads,dropout)
        self.norm2=RMSNorm(d_model,rms_eps); self.ff=SwiGLU(d_model,ffn_dim,dropout); self.resid_drop=nn.Dropout(dropout)
    def forward(self,x,cos,sin,past_kv=None,use_cache=False,key_valid=None):
        a,new_kv=self.attn(self.norm1(x),cos,sin,past_kv=past_kv,use_cache=use_cache,key_valid=key_valid)
        x=x+self.resid_drop(a); x=x+self.ff(self.norm2(x)); return x,new_kv

class EfficientDecoderLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,n_kv_heads,ffn_dim,dropout,max_positions,rope_base,rms_eps):
        super().__init__(); self.vocab_size=vocab_size; self.d_model=d_model; self.max_positions=max_positions
        self.token_emb=nn.Embedding(vocab_size,d_model); self.drop=nn.Dropout(dropout)
        self.rope=RotaryTable(d_model//n_heads,max_positions,rope_base)
        self.blocks=nn.ModuleList([EfficientDecoderBlock(d_model,n_heads,n_kv_heads,ffn_dim,dropout,rms_eps) for _ in range(n_layers)])
        self.final_norm=RMSNorm(d_model,rms_eps)
    def forward_hidden(self,input_ids,past_key_values=None,use_cache=False,position_ids=None,key_valid=None):
        B,T=input_ids.shape; past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len+T>self.max_positions: raise ValueError(f"context {past_len+T} exceeds max_positions={self.max_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        cos,sin=self.rope(position_ids); x=self.drop(self.token_emb(input_ids)); new_cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,cos,sin,past_kv=past,use_cache=use_cache,key_valid=key_valid)
            if use_cache: new_cache.append(kv)
        return self.final_norm(x),new_cache
    def project(self,hidden): return F.linear(hidden,self.token_emb.weight)

class Model4ForNER(nn.Module):
    def __init__(self,backbone,n_tags,dropout=0.1):
        super().__init__(); self.backbone=backbone; self.ner_dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(backbone.d_model,n_tags)
    def forward(self,input_ids,attention_mask=None):
        h,_=self.backbone.forward_hidden(input_ids,key_valid=attention_mask)
        return self.classifier(self.ner_dropout(h))


# ===== Model 2 student: exact Phase-1 decoder-only architecture =====
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_positions):
        super().__init__()
        pos=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_positions,d_model,dtype=torch.float32)
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe,persistent=False)
    def forward(self,position_ids):
        return self.pe[position_ids]

class M2CausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__(); assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout

    def forward(self,x,past_kv=None,use_cache=False,key_valid=None):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)

        past_len=0
        if past_kv is not None:
            pk,pv=past_kv; past_len=pk.size(2)
            k=torch.cat([pk,k],dim=2); v=torch.cat([pv,v],dim=2)

        attn_mask=None; is_causal=False
        if key_valid is not None:
            key_mask=key_valid[:,None,None,:].bool()
            if past_len==0 and T>1:
                causal=torch.ones((T,T),dtype=torch.bool,device=x.device).tril()
                attn_mask=key_mask & causal[None,None,:,:]
                eye=torch.eye(T,dtype=torch.bool,device=x.device)[None,None,:,:]
                attn_mask=attn_mask | eye
            else:
                attn_mask=key_mask
        elif past_kv is None:
            is_causal=True

        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.out_proj(y)
        return y,((k,v) if use_cache else None)

class M2DecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.attn=M2CausalSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout)
        )
        self.resid_drop=nn.Dropout(dropout)
    def forward(self,x,past_kv=None,use_cache=False,key_valid=None):
        a,new_kv=self.attn(self.ln1(x),past_kv=past_kv,use_cache=use_cache,key_valid=key_valid)
        x=x+self.resid_drop(a)
        x=x+self.ff(self.ln2(x))
        return x,new_kv

class DecoderOnlyLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_layers,n_heads,ffn_dim,dropout,max_positions):
        super().__init__()
        self.vocab_size=vocab_size; self.d_model=d_model; self.max_positions=max_positions
        self.token_emb=nn.Embedding(vocab_size,d_model)
        self.pos=SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([
            M2DecoderBlock(d_model,n_heads,ffn_dim,dropout) for _ in range(n_layers)
        ])
        self.final_ln=nn.LayerNorm(d_model)
        self.apply(self._init_weights)
        scaled_std=0.02/math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight,mean=0.0,std=scaled_std)
            nn.init.normal_(block.ff[2].weight,mean=0.0,std=scaled_std)

    def _init_weights(self,m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)

    def forward_hidden(self,input_ids,past_key_values=None,use_cache=False,position_ids=None,key_valid=None):
        B,T=input_ids.shape
        past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len+T>self.max_positions:
            raise ValueError(f"context {past_len+T} exceeds max_positions={self.max_positions}")
        if position_ids is None:
            position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        x=self.token_emb(input_ids)+self.pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        new_cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,past_kv=past,use_cache=use_cache,key_valid=key_valid)
            if use_cache: new_cache.append(kv)
        return self.final_ln(x),new_cache

    def project(self,hidden):
        return F.linear(hidden,self.token_emb.weight)

class Model2KDStudent(nn.Module):
    def __init__(self,backbone,n_tags,dropout=0.1,hidden=256):
        super().__init__()
        self.backbone=backbone
        self.task_norm=nn.LayerNorm(backbone.d_model)
        self.bigru=nn.GRU(
            input_size=backbone.d_model,hidden_size=hidden,
            num_layers=1,batch_first=True,bidirectional=True
        )
        self.drop=nn.Dropout(dropout)
        self.classifier=nn.Linear(2*hidden,n_tags)

    def forward(self,input_ids,attention_mask=None):
        key_valid=None if attention_mask is None else attention_mask.bool()
        h,_=self.backbone.forward_hidden(input_ids,key_valid=key_valid)
        h=self.task_norm(h)
        T=h.size(1)
        if attention_mask is not None:
            lengths=attention_mask.sum(1).clamp(min=1).cpu()
            packed=nn.utils.rnn.pack_padded_sequence(h,lengths,batch_first=True,enforce_sorted=False)
            out,_=self.bigru(packed)
            h,_=nn.utils.rnn.pad_packed_sequence(out,batch_first=True,total_length=T)
        else:
            h,_=self.bigru(h)
        return self.classifier(self.drop(h))

    def head_parameters(self):
        return list(self.task_norm.parameters())+list(self.bigru.parameters())+list(self.classifier.parameters())


In [12]:
# ===== Reconstruct frozen Exp5 teacher =====
m4_obj=torch.load(MODEL4_PHASE1_PATH,map_location="cpu",weights_only=False)
m4_cfg=m4_obj["config"]
assert m4_cfg.get("architecture")=="efficient_gqa_rope_swiglu_decoder"
assert int(m4_cfg["vocab_size"])==len(tokenizer)

# Read exact Exp5 settings from teacher checkpoint when available.
teacher_obj=torch.load(TEACHER_HELDOUT_PATH,map_location="cpu",weights_only=False)
TEACHER_TOP_N=int(teacher_obj.get("top_n_unfrozen",6))
TEACHER_FIRST=int(teacher_obj.get("first_trainable_block",17-TEACHER_TOP_N))
TEACHER_BIDIR=bool(teacher_obj.get("bidirectional",True))
TEACHER_BIDIR_ONLY=bool(teacher_obj.get("bidir_only_in_trainable",True))

def build_m4_backbone():
    bb=EfficientDecoderLM(
        vocab_size=int(m4_cfg["vocab_size"]),d_model=int(m4_cfg["d_model"]),
        n_layers=int(m4_cfg["n_layers"]),n_heads=int(m4_cfg["n_heads"]),
        n_kv_heads=int(m4_cfg["n_kv_heads"]),ffn_dim=int(m4_cfg["ffn_dim"]),
        dropout=float(m4_cfg.get("dropout",0.0)),max_positions=int(m4_cfg["max_positions"]),
        rope_base=float(m4_cfg.get("rope_base",10000.0)),rms_eps=float(m4_cfg.get("rms_eps",1e-5))
    )
    bb.load_state_dict(m4_obj["model"],strict=True)
    return bb

def configure_teacher_structure(m):
    # Reproduce the Exp5 trainable-name structure so trainable_state keys load exactly.
    for p in m.parameters(): p.requires_grad=False
    n=len(m.backbone.blocks); first=n-TEACHER_TOP_N
    for i in range(first,n):
        for p in m.backbone.blocks[i].parameters(): p.requires_grad=True
    for p in m.backbone.final_norm.parameters(): p.requires_grad=True
    for p in m.classifier.parameters(): p.requires_grad=True

    for i,blk in enumerate(m.backbone.blocks):
        if TEACHER_BIDIR_ONLY and i<first: blk.attn.bidir_override=False
        else: blk.attn.bidir_override=TEACHER_BIDIR
    return first

def load_named_trainable_state(m,state):
    own=m.state_dict()
    expected={n for n,p in m.named_parameters() if p.requires_grad}
    incoming=set(state)
    unexpected=sorted(incoming-set(own))
    missing=sorted(expected-incoming)
    if unexpected:
        raise KeyError(f"Unexpected teacher keys: {unexpected[:10]}")
    if missing:
        raise KeyError(f"Missing teacher trainable keys: {missing[:10]}")
    for k,v in state.items():
        own[k].copy_(v.to(own[k].device,dtype=own[k].dtype))

teacher=Model4ForNER(build_m4_backbone(),N_TAGS,HEAD_DROPOUT)
first_teacher=configure_teacher_structure(teacher)
load_named_trainable_state(teacher,teacher_obj["trainable_state"])
for p in teacher.parameters(): p.requires_grad=False
teacher=teacher.to(DEVICE).eval()

print("✅ Exp5 teacher reconstructed")
print(" Teacher blocks causal:",list(range(first_teacher)))
print(" Teacher blocks bidirectional:",list(range(first_teacher,17)))
print(" Teacher heldout F1 in checkpoint:",teacher_obj.get("strict_macro_f1"))

# ===== Clean Phase-1 Model-2 student =====
m2_obj=torch.load(MODEL2_PHASE1_PATH,map_location="cpu",weights_only=False)
m2_cfg=m2_obj.get("config",{})
V2=int(m2_cfg.get("vocab_size",24576)); D2=int(m2_cfg.get("d_model",640))
L2=int(m2_cfg.get("n_layers",13)); H2=int(m2_cfg.get("n_heads",10))
FF2=int(m2_cfg.get("ffn_dim",2560)); DROP2=float(m2_cfg.get("dropout",0.0))
MP2=int(m2_cfg.get("max_positions",2048))
assert V2==len(tokenizer)
assert (V2,D2,L2,H2,FF2)==(24576,640,13,10,2560)

def build_m2_backbone():
    bb=DecoderOnlyLM(V2,D2,L2,H2,FF2,DROP2,MP2)
    bb.load_state_dict(m2_obj["model"],strict=True)
    return bb

student=Model2KDStudent(build_m2_backbone(),N_TAGS,HEAD_DROPOUT,HEAD_HIDDEN).to(DEVICE)
print("✅ Phase-1 Model2 student loaded STRICTLY")

teacher_params=sum(p.numel() for p in teacher.parameters())
student_params=sum(p.numel() for p in student.parameters())
student_trainable=sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"Teacher params: {teacher_params/1e6:.2f}M (frozen)")
print(f"Student params: {student_params/1e6:.2f}M | trainable={student_trainable/1e6:.2f}M")
print("Student backbone attention: native causal")
print("Student right-context mechanism: BiGRU task head")


✅ Exp5 teacher reconstructed
 Teacher blocks causal: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
 Teacher blocks bidirectional: [11, 12, 13, 14, 15, 16]
 Teacher heldout F1 in checkpoint: 0.46060552210605116
✅ Phase-1 Model2 student loaded STRICTLY
Teacher params: 99.34M (frozen)
Student params: 81.14M | trainable=81.14M
Student backbone attention: native causal
Student right-context mechanism: BiGRU task head


In [13]:
# ===== Teacher/student alignment sanity =====
assert AMP is False,"Keep KD in FP32 for the first official run."

@torch.no_grad()
def kd_sanity(ds,n=8):
    teacher.eval(); student.eval()
    rows=[]
    for i in range(min(n,len(ds))):
        item=ds[i]
        ids=item["input_ids"][None,:].to(DEVICE)
        mask=torch.ones_like(ids,dtype=torch.bool)
        t=teacher(ids,mask)
        s=student(ids,mask)
        rows.append((tuple(t.shape),tuple(s.shape),bool(torch.isfinite(t).all()),bool(torch.isfinite(s).all())))
    return rows

rows=kd_sanity(train_ds)
print("KD shape/finite sanity:",rows[:3])
assert rows and all(a==b and c and d for a,b,c,d in rows)
assert rows[0][0][-1]==N_TAGS
print("✅ Teacher and student produce perfectly aligned token logits with the same tag dimension.")


KD shape/finite sanity: [((1, 768, 49), (1, 768, 49), True, True), ((1, 768, 49), (1, 768, 49), True, True), ((1, 768, 49), (1, 768, 49), True, True)]
✅ Teacher and student produce perfectly aligned token logits with the same tag dimension.


In [14]:
# ===== Class weights + student optimizer =====
def count_training_tags(ds):
    c=Counter()
    for d in ds.docs.values():
        if d["tags"] is not None:
            c.update(d["tags"])
    return c

def make_class_weights(ds):
    if not USE_CLASS_WEIGHTS:
        return None
    tag_counts=count_training_tags(ds)
    counts=np.array([max(1,tag_counts.get(i,0)) for i in range(N_TAGS)],dtype=np.float64)
    w=np.sqrt(counts.sum()/counts)
    w/=np.median(w[1:]) if len(w)>1 else 1.0
    w=np.clip(w,0.20,5.0)
    w[O_ID]=min(w[O_ID],0.35)
    return torch.tensor(w,dtype=torch.float32,device=DEVICE)

tag_counts=count_training_tags(train_ds)
CLASS_WEIGHTS=make_class_weights(train_ds)

print("\nTraining tag statistics:")
for tid in range(N_TAGS):
    if tag_counts.get(tid,0)>0:
        wt=float(CLASS_WEIGHTS[tid].detach().cpu()) if CLASS_WEIGHTS is not None else 1.0
        print(f"{ID2TAG[tid]:16s} count={tag_counts.get(tid,0):8d} weight={wt:.3f}")

train_loader=DataLoader(
    train_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,
    collate_fn=collate_windows,num_workers=2,pin_memory=True
)
val_loader=DataLoader(
    val_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,
    collate_fn=collate_windows,num_workers=2,pin_memory=True
)

def make_student_param_groups(m):
    groups=[]
    n=len(m.backbone.blocks)

    # Preserve lower Phase-1 features more strongly; top block gets STUDENT_BACKBONE_LR.
    emb_lr=STUDENT_BACKBONE_LR*(STUDENT_LAYERWISE_LR_DECAY**n)
    groups.append({
        "params":list(m.backbone.token_emb.parameters()),
        "lr":emb_lr,"name":"token_emb"
    })

    for i,blk in enumerate(m.backbone.blocks):
        depth_from_top=(n-1)-i
        lr=STUDENT_BACKBONE_LR*(STUDENT_LAYERWISE_LR_DECAY**depth_from_top)
        groups.append({
            "params":list(blk.parameters()),
            "lr":lr,"name":f"block_{i}"
        })

    groups.append({
        "params":list(m.backbone.final_ln.parameters()),
        "lr":STUDENT_BACKBONE_LR,"name":"final_ln"
    })
    groups.append({
        "params":m.head_parameters(),
        "lr":STUDENT_HEAD_LR,"name":"ner_head"
    })
    return groups

student_groups=make_student_param_groups(student)
optimizer=torch.optim.AdamW(student_groups,weight_decay=WEIGHT_DECAY)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=max(1,updates_per_epoch*EPOCHS)
warmup_updates=max(1,int(WARMUP_RATIO*total_updates))

def lr_lambda(step):
    if step<warmup_updates:
        return (step+1)/warmup_updates
    p=(step-warmup_updates)/max(1,total_updates-warmup_updates)
    return 0.5*(1+math.cos(math.pi*min(1.0,p)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
trainable_params=[p for p in student.parameters() if p.requires_grad]

print("\nBatches/epoch:",len(train_loader),"updates/epoch:",updates_per_epoch,"total updates:",total_updates)
print("Student optimizer groups:")
for g in optimizer.param_groups:
    print(f"  {g.get('name','?'):12s} base_lr={g['initial_lr']:.3e} params={sum(p.numel() for p in g['params'])/1e6:.3f}M")



Training tag statistics:
O                count=  390766 weight=0.200
B-APP            count=     169 weight=1.015
I-APP            count=     536 weight=0.570
L-APP            count=     169 weight=1.015
U-APP            count=      34 weight=2.262
B-RESP           count=     174 weight=1.000
I-RESP           count=     498 weight=0.591
L-RESP           count=     174 weight=1.000
U-RESP           count=      17 weight=3.199
B-A.COUNSEL      count=      64 weight=1.649
I-A.COUNSEL      count=     275 weight=0.795
L-A.COUNSEL      count=      64 weight=1.649
U-A.COUNSEL      count=       6 weight=5.000
B-R.COUNSEL      count=      83 weight=1.448
I-R.COUNSEL      count=     295 weight=0.768
L-R.COUNSEL      count=      83 weight=1.448
U-R.COUNSEL      count=       4 weight=5.000
B-JUDGE          count=     118 weight=1.214
I-JUDGE          count=     574 weight=0.551
L-JUDGE          count=     118 weight=1.214
U-JUDGE          count=       4 weight=5.000
B-WIT            count=      

In [15]:
def strict_scores(truth,pred,labels=LABELS,verbose=True):
    if set(truth)!=set(pred): raise ValueError("ID mismatch")
    stats={l:[0,0,0] for l in labels}; allowed=set(labels)
    def ss(xs):
        out=set()
        for s in xs or []:
            a,b,l=s[:3]
            if not isinstance(a,int) or isinstance(a,bool) or not isinstance(b,int) or isinstance(b,bool): raise ValueError("bad offsets")
            if a<0 or b<=a or l not in allowed: raise ValueError("bad span")
            t=(a,b,l)
            if t in out: raise ValueError("duplicate span")
            out.add(t)
        return out
    for did in truth:
        g=ss(truth[did]); p=ss(pred[did])
        for l in labels:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    ps=[]; rs=[]; fs=[]; per={}
    for l in labels:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        f=2*pr*rc/(pr+rc) if pr+rc else 0
        ps.append(pr); rs.append(rc); fs.append(f); per[l]=(pr,rc,f,tp,fp,fn)
        if verbose: print(f"{l:10s}: P={pr:.6f} R={rc:.6f} F1={f:.6f}")
    out={"macro_f1":float(np.mean(fs)),"macro_precision":float(np.mean(ps)),"macro_recall":float(np.mean(rs)),"per_label":per}
    if verbose:
        print(f"Strict Macro-F1:        {out['macro_f1']:.6f}")
        print(f"Strict Macro-Precision: {out['macro_precision']:.6f}")
        print(f"Strict Macro-Recall:    {out['macro_recall']:.6f}")
    return out

def gold_for_eval(gold_map):
    out={}
    for did,ents in gold_map.items():
        seen=set(); lst=[]
        for a,b,l,et in ents:
            t=(int(a),int(b),str(l))
            if t not in seen: seen.add(t); lst.append(list(t))      # evaluator rejects duplicate spans
        out[did]=lst
    return out

# ---- constrained BILOU Viterbi ----
def _build_transitions():
    pref=[None]+[t.split("-",1)[0] for t in TAG_NAMES[1:]]
    lab=[None]+[t.split("-",1)[1] for t in TAG_NAMES[1:]]
    allow=np.zeros((N_TAGS,N_TAGS),dtype=bool)
    for a in range(N_TAGS):
        a_open = pref[a] in ("B","I")
        for b in range(N_TAGS):
            if a_open: allow[a,b]=(pref[b] in ("I","L") and lab[b]==lab[a])
            else:      allow[a,b]=(b==O_ID or pref[b] in ("B","U"))
    start=np.array([ (i==O_ID or pref[i] in ("B","U")) for i in range(N_TAGS)])
    end=np.array([ (i==O_ID or pref[i] in ("L","U")) for i in range(N_TAGS)])
    return allow,start,end
TRANS_ALLOW,START_OK,END_OK=_build_transitions()
NEG=-1e9

def softmax_np(x):
    x=x-x.max(-1,keepdims=True); e=np.exp(x); return e/e.sum(-1,keepdims=True)

def viterbi_path(logp):
    n=logp.shape[0]
    trans=np.where(TRANS_ALLOW,0.0,NEG).astype(np.float32)
    score=np.where(START_OK,logp[0],NEG).astype(np.float32)
    back=np.zeros((n,N_TAGS),dtype=np.int16)
    for t in range(1,n):
        cand=score[:,None]+trans                     # (from,to)
        bp=cand.argmax(0); score=cand[bp,np.arange(N_TAGS)]+logp[t]; back[t]=bp
    score=np.where(END_OK,score,NEG)
    path=np.zeros(n,dtype=np.int64); path[-1]=int(score.argmax())
    for t in range(n-1,0,-1): path[t-1]=back[t,path[t]]
    return path

def decode_spans(doc,logits,viterbi=None):
    """Returns [(start_char,end_char,label,confidence)], confidence = min token prob of the span's tags."""
    if len(logits)==0: return []
    viterbi=USE_VITERBI if viterbi is None else viterbi
    logits=np.asarray(logits,dtype=np.float32); probs=softmax_np(logits)
    if viterbi: pred=viterbi_path(np.log(probs+1e-9))
    else:       pred=probs.argmax(-1)
    conf=probs[np.arange(len(pred)),pred]; offsets=doc["offsets"]; text=doc["text"]; out=[]; i=0; n=len(pred)
    while i<n:
        tag=ID2TAG[int(pred[i])]
        if tag=="O": i+=1; continue
        pf,lab=tag.split("-",1)
        if pf=="U": j=i
        elif pf=="B":
            j=i+1
            while j<n and ID2TAG[int(pred[j])]==f"I-{lab}": j+=1
            if not(j<n and ID2TAG[int(pred[j])]==f"L-{lab}"): i+=1; continue     # malformed (argmax mode only)
        else: i+=1; continue
        a=offsets[i][0]; b=offsets[j][1]
        if 0<=a<b<=len(text) and text[a:b].strip(): out.append((int(a),int(b),lab,float(conf[i:j+1].min())))
        i=j+1
    return out

def _threshold_for_label(threshold,label):
    if isinstance(threshold,dict):
        return float(threshold.get(label,0.0))
    return float(threshold)

def filter_spans(spans,threshold=0.0):
    """threshold may be a scalar or {label: threshold} dict."""
    seen=set(); res=[]
    for a,b,l,c in spans:
        th=_threshold_for_label(threshold,l)
        if c>=th and (a,b,l) not in seen:
            seen.add((a,b,l)); res.append([a,b,l])
    return res

def decode_bilou_from_logits(doc,logits,threshold=0.0,viterbi=None):
    return filter_spans(decode_spans(doc,logits,viterbi),threshold)

def tune_global_threshold(span_cache,truth,grid):
    rows=[]
    for th in grid:
        pred={did:filter_spans(span_cache[did],th) for did in span_cache}
        sc=strict_scores(truth,pred,verbose=False)
        rows.append((sc["macro_f1"],float(th),sc))
    rows.sort(reverse=True,key=lambda x:x[0])
    return rows[0],rows

def tune_per_label_thresholds(span_cache,truth,grid):
    """
    Macro-F1 is the unweighted mean of per-label F1 values.
    For a fixed decoded span set, each label's confidence threshold affects only
    that label's F1, so independent per-label tuning directly optimizes this
    held-out Macro-F1 threshold family.
    """
    thresholds={}
    details={}
    for lab in LABELS:
        best=(-1.0,0.0,None)
        for th in grid:
            pred={}
            for did,spans in span_cache.items():
                pred[did]=[[a,b,l] for a,b,l,c in spans if l==lab and c>=float(th)]
            # Score only this label using exact strict matching.
            sc=strict_scores(
                {did:[s for s in truth[did] if s[2]==lab] for did in truth},
                pred,
                labels=[lab],
                verbose=False,
            )
            row=(sc["macro_f1"],float(th),sc)
            if row[0]>best[0]:
                best=row
        thresholds[lab]=best[1]
        details[lab]=best[2]

    combined={did:filter_spans(span_cache[did],thresholds) for did in span_cache}
    combined_sc=strict_scores(truth,combined,verbose=False)
    return thresholds,combined_sc,details


In [16]:
# ===== ORACLE CEILING TEST: gold tags -> decode -> strict F1 (no model involved) =====
# This is the best Strict Macro-F1 your tokenizer/alignment can possibly give on Fold 2.
# If it is far below ~0.95, token boundaries do not line up with gold spans (try PRESPLIT_PUNCT=True and compare).
def oracle_logits(doc):
    lg=np.full((len(doc["ids"]),N_TAGS),-10.0,dtype=np.float32)
    for k,t in enumerate(doc["tags"]): lg[k,t]=10.0
    return lg
oracle_pred={did:decode_bilou_from_logits(d,oracle_logits(d),0.0) for did,d in val_ds.docs.items()}
_truth_val=gold_for_eval(fold2_gold)
print("ORACLE (Fold 2):"); oracle_sc=strict_scores(_truth_val,oracle_pred,verbose=True)


ORACLE (Fold 2):
APP       : P=1.000000 R=0.995000 F1=0.997494
RESP      : P=1.000000 R=1.000000 F1=1.000000
A.COUNSEL : P=1.000000 R=1.000000 F1=1.000000
R.COUNSEL : P=1.000000 R=1.000000 F1=1.000000
JUDGE     : P=1.000000 R=1.000000 F1=1.000000
WIT       : P=1.000000 R=1.000000 F1=1.000000
AUTH      : P=1.000000 R=1.000000 F1=1.000000
COURT     : P=1.000000 R=0.980287 F1=0.990045
STAT      : P=1.000000 R=0.994558 F1=0.997271
PREC      : P=1.000000 R=1.000000 F1=1.000000
DATE      : P=1.000000 R=1.000000 F1=1.000000
CASENO    : P=1.000000 R=1.000000 F1=1.000000
Strict Macro-F1:        0.998734
Strict Macro-Precision: 1.000000
Strict Macro-Recall:    0.997487


In [17]:
@torch.no_grad()
def infer_dataset(model,ds,loader,threshold=0.0,return_logits=False):
    model.eval()
    doc_logits={did:np.zeros((len(d["ids"]),N_TAGS),dtype=np.float32) for did,d in ds.docs.items()}
    best_dist={did:np.full(len(d["ids"]),-1.0,dtype=np.float32) for did,d in ds.docs.items()}   # centrality of the chosen logits
    for bi,batch in enumerate(loader):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(AMP and DEVICE=="cuda")):
            logits=model(ids,mask)
        logits=logits.float().cpu().numpy()
        for r,did in enumerate(batch["doc_ids"]):
            s=batch["start_tokens"][r]; e=batch["end_tokens"][r]; n=len(ds.docs[did]["ids"])
            if e<=s: continue
            L=e-s; loc=np.arange(L)
            dl=loc if s>0 else np.full(L,1e9); dr=(L-1-loc) if e<n else np.full(L,1e9)   # no penalty at true doc borders
            dist=np.minimum(dl,dr).astype(np.float32)
            upd=dist>best_dist[did][s:e]
            doc_logits[did][s:e][upd]=logits[r,:L][upd]; best_dist[did][s:e][upd]=dist[upd]
        if (bi+1)%50==0: print(f"inference batch {bi+1}/{len(loader)}",flush=True)
    for did in best_dist:
        if len(best_dist[did]) and (best_dist[did]<0).any(): raise RuntimeError(f"{did}: unfilled tokens")
    pred={did:decode_bilou_from_logits(ds.docs[did],doc_logits[did],threshold) for did in ds.docs}
    return (pred,doc_logits) if return_logits else pred


In [18]:
# ===== Teacher sanity + reusable teacher-logit cache =====
if CHECK_TEACHER_F1:
    saved_teacher_th=teacher_obj.get("best_threshold",0.0)
    _t_pred,_t_logits=infer_dataset(teacher,val_ds,val_loader,saved_teacher_th,True)
    _t_sc=strict_scores(gold_for_eval(fold2_gold),_t_pred,verbose=False)
    _stored=teacher_obj.get("strict_macro_f1")
    print(f"Teacher Fold-2 Macro-F1 using checkpoint thresholds: {_t_sc['macro_f1']:.6f}")
    print("Teacher F1 stored in checkpoint:",_stored)
    if _stored is not None:
        delta=abs(float(_stored)-float(_t_sc["macro_f1"]))
        print(f"Teacher reconstruction |ΔF1|={delta:.8f}")
        assert delta<5e-4, (
            "Teacher reconstruction does not reproduce the Exp5 checkpoint. "
            "Stop here and inspect architecture/checkpoint/tokenizer."
        )
    assert _t_sc["macro_f1"]>0.30,"Teacher reconstruction looks broken (Fold-2 F1 < 0.30)"
    del _t_pred,_t_logits
    gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()

def _cache_key(did,s,e):
    return (str(did),int(s),int(e))

@torch.no_grad()
def precompute_teacher_logits(model,ds,batch_size=EVAL_BATCH_SIZE):
    """Run teacher exactly once per window and keep compact logits on CPU."""
    loader=DataLoader(
        ds,batch_size=batch_size,shuffle=False,
        collate_fn=collate_windows,num_workers=2,pin_memory=True
    )
    model.eval()
    cache={}
    dtype=np.float16 if TEACHER_CACHE_DTYPE=="float16" else np.float32
    t0=time.time()
    for bi,batch in enumerate(loader,1):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        logits=model(ids,mask).float().cpu().numpy()
        for r,did in enumerate(batch["doc_ids"]):
            s=int(batch["start_tokens"][r]); e=int(batch["end_tokens"][r]); L=e-s
            key=_cache_key(did,s,e)
            if key in cache:
                raise RuntimeError(f"Duplicate teacher-cache key: {key}")
            cache[key]=logits[r,:L].astype(dtype,copy=True)
        if bi%50==0 or bi==len(loader):
            print(f"teacher cache {bi}/{len(loader)}",flush=True)
    expected={_cache_key(did,s,e) for did,s,e in ds.windows}
    assert set(cache)==expected
    mb=sum(x.nbytes for x in cache.values())/1024**2
    print(f"✅ Teacher logits cached: {len(cache)} windows, {mb:.1f} MiB, {(time.time()-t0)/60:.1f} min")
    return cache

def teacher_logits_from_cache(batch,cache,device):
    B=len(batch["doc_ids"])
    T=batch["input_ids"].shape[1]
    out=torch.zeros((B,T,N_TAGS),dtype=torch.float32,device=device)
    for r,did in enumerate(batch["doc_ids"]):
        s=int(batch["start_tokens"][r]); e=int(batch["end_tokens"][r]); L=e-s
        arr=cache[_cache_key(did,s,e)]
        assert arr.shape==(L,N_TAGS)
        if L:
            out[r,:L]=torch.from_numpy(arr.astype(np.float32,copy=False)).to(device,non_blocking=True)
    return out

TRAIN_TEACHER_CACHE=None
if CACHE_TEACHER_LOGITS:
    print("\nCaching Fold-1 teacher logits once...")
    TRAIN_TEACHER_CACHE=precompute_teacher_logits(teacher,train_ds)
    # Teacher is no longer needed on GPU during student optimization.
    teacher=teacher.cpu()
    gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()
    print("✅ Held-out teacher moved to CPU; GPU memory is now dedicated to the student.")


inference batch 50/374
inference batch 100/374
inference batch 150/374
inference batch 200/374
inference batch 250/374
inference batch 300/374
inference batch 350/374
Teacher Fold-2 Macro-F1 using checkpoint thresholds: 0.460606
Teacher F1 stored in checkpoint: 0.46060552210605116
Teacher reconstruction |ΔF1|=0.00000000

Caching Fold-1 teacher logits once...
teacher cache 50/368
teacher cache 100/368
teacher cache 150/368
teacher cache 200/368
teacher cache 250/368
teacher cache 300/368
teacher cache 350/368
teacher cache 368/368
✅ Teacher logits cached: 736 windows, 51.9 MiB, 0.8 min
✅ Held-out teacher moved to CPU; GPU memory is now dedicated to the student.


In [19]:
def hard_ner_loss(logits,labels):
    return F.cross_entropy(
        logits.float().reshape(-1,N_TAGS),labels.reshape(-1),
        weight=CLASS_WEIGHTS,ignore_index=IGNORE_INDEX
    )

def kd_kl_loss(student_logits,teacher_logits,labels,T=2.0,class_weights=None):
    # Token-level KD on valid supervised positions only.
    s=student_logits.float().reshape(-1,N_TAGS)
    t=teacher_logits.float().reshape(-1,N_TAGS)
    y=labels.reshape(-1)
    keep=y.ne(IGNORE_INDEX)
    if not bool(keep.any()):
        return s.sum()*0.0
    s=s[keep]/T; t=t[keep]/T; yv=y[keep]
    log_ps=F.log_softmax(s,dim=-1)
    pt=F.softmax(t,dim=-1)
    per_tok=F.kl_div(log_ps,pt,reduction="none").sum(-1)
    if KD_WEIGHT_BY_CLASS and class_weights is not None:
        wt=class_weights[yv].float()
        # Normalize so the KD scale stays comparable across batches.
        wt=wt/wt.mean().clamp_min(1e-8)
        per_tok=per_tok*wt
    return per_tok.mean()*(T*T)

def student_state_dict(m):
    return {k:v.detach().cpu() for k,v in m.state_dict().items()}

best_f1=-1.0
best_epoch=None
epochs_without_improve=0
history=[]
truth_val=gold_for_eval(fold2_gold)
GOLD_NONO=float(np.mean(np.concatenate([
    np.array(d["tags"])!=O_ID for d in val_ds.docs.values() if len(d["tags"])
])))

print("🔥 ACTUAL KNOWLEDGE DISTILLATION STARTS HERE")
print(f"KD: hard={HARD_LOSS_WEIGHT:.2f}, soft={KD_LOSS_WEIGHT:.2f}, T={KD_TEMPERATURE}")

for epoch in range(1,EPOCHS+1):
    student.train()
    optimizer.zero_grad(set_to_none=True)
    run_total=run_hard=run_kd=0.0; nbatches=0; t0=time.time()

    for step,batch in enumerate(train_loader,1):
        ids=batch["input_ids"].to(DEVICE,non_blocking=True)
        mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
        y=batch["labels"].to(DEVICE,non_blocking=True)

        if TRAIN_TEACHER_CACHE is not None:
            t_logits=teacher_logits_from_cache(batch,TRAIN_TEACHER_CACHE,DEVICE)
        else:
            teacher=teacher.to(DEVICE).eval()
            with torch.no_grad():
                t_logits=teacher(ids,mask)

        s_logits=student(ids,mask)
        hard=hard_ner_loss(s_logits,y)
        soft=kd_kl_loss(s_logits,t_logits,y,KD_TEMPERATURE,CLASS_WEIGHTS)
        loss=(HARD_LOSS_WEIGHT*hard + KD_LOSS_WEIGHT*soft)/GRAD_ACCUM_STEPS

        if not torch.isfinite(loss.detach()):
            raise FloatingPointError(f"Non-finite KD loss at epoch={epoch}, batch={step}")

        loss.backward()
        run_total+=float(loss.detach().cpu())*GRAD_ACCUM_STEPS
        run_hard+=float(hard.detach().cpu())
        run_kd+=float(soft.detach().cpu())
        nbatches+=1

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            grad_norm=torch.nn.utils.clip_grad_norm_(trainable_params,MAX_GRAD_NORM)
            if not torch.isfinite(torch.as_tensor(grad_norm)):
                raise FloatingPointError(f"Non-finite student grad norm at epoch={epoch}, batch={step}")
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if step%100==0 or step==len(train_loader):
            print(
                f"epoch {epoch}/{EPOCHS} step {step}/{len(train_loader)} "
                f"loss={run_total/max(1,nbatches):.4f} hard={run_hard/max(1,nbatches):.4f} "
                f"kd={run_kd/max(1,nbatches):.4f} bb_lr={optimizer.param_groups[0]['lr']:.2e} "
                f"elapsed={(time.time()-t0)/60:.1f}m",flush=True
            )

    _,val_logits=infer_dataset(student,val_ds,val_loader,0.0,True)
    _pn=float(np.mean(np.concatenate([lg.argmax(-1)!=O_ID for lg in val_logits.values() if len(lg)])))
    print(f"   diagnostic: student predicted non-O={_pn:.4f} (gold={GOLD_NONO:.4f})")

    span_cache={did:decode_spans(val_ds.docs[did],val_logits[did]) for did in val_ds.docs}
    (global_f1,global_th,global_sc),_=tune_global_threshold(span_cache,truth_val,THRESHOLD_GRID)
    if USE_PER_LABEL_THRESHOLDS:
        epoch_th,epoch_sc,_=tune_per_label_thresholds(span_cache,truth_val,THRESHOLD_GRID)
        epoch_f1=epoch_sc["macro_f1"]
    else:
        epoch_th=float(global_th); epoch_sc=global_sc; epoch_f1=global_f1

    print(
        f"Epoch {epoch}: global Macro-F1={global_f1:.6f} @ {global_th:.2f} | "
        f"selected Macro-F1={epoch_f1:.6f} P={epoch_sc['macro_precision']:.6f} "
        f"R={epoch_sc['macro_recall']:.6f}"
    )
    if isinstance(epoch_th,dict):
        print("   thresholds:"," ".join(f"{l}={epoch_th[l]:.2f}" for l in LABELS))

    hist={
        "epoch":epoch,
        "train_loss":run_total/max(1,nbatches),
        "hard_loss":run_hard/max(1,nbatches),
        "kd_loss":run_kd/max(1,nbatches),
        "macro_f1":float(epoch_f1),
        "macro_precision":float(epoch_sc["macro_precision"]),
        "macro_recall":float(epoch_sc["macro_recall"]),
        "threshold":epoch_th,
    }
    history.append(hist)

    improved=epoch_f1>best_f1
    if improved:
        best_f1=epoch_f1; best_epoch=epoch; epochs_without_improve=0
        torch.save({
            "student_state":student_state_dict(student),
            "student_phase1_checkpoint":MODEL2_PHASE1_PATH,
            "teacher_phase1_checkpoint":MODEL4_PHASE1_PATH,
            "teacher_phase2_checkpoint":TEACHER_HELDOUT_PATH,
            "best_epoch":best_epoch,
            "best_threshold":epoch_th,
            "strict_macro_f1":float(epoch_f1),
            "history":history,
            "kd_temperature":KD_TEMPERATURE,
            "hard_loss_weight":HARD_LOSS_WEIGHT,
            "kd_loss_weight":KD_LOSS_WEIGHT,
            "experiment":"Phase2 Exp6 KD Model4-teacher -> Model2-student",
            "tokenizer_sha256":sha256_file(TOKENIZER_PATH),
        },BEST_KD_PATH)
        print("✅ New best KD student saved:",BEST_KD_PATH)
        print("   per-label F1:"," ".join(f"{l}={v[2]:.3f}" for l,v in epoch_sc["per_label"].items()))
    else:
        epochs_without_improve+=1

    print("-"*90)
    if epoch>=3 and best_f1<0.05:
        print("❌ KD student still below 0.05 after 3 epochs — stop and inspect.")
        break
    if epochs_without_improve>=EARLY_STOP_PATIENCE:
        print(f"⏹️ Early stopping after {EARLY_STOP_PATIENCE} non-improving epochs.")
        break

print("Best epoch:",best_epoch,"Best held-out Strict Macro-F1:",best_f1)


🔥 ACTUAL KNOWLEDGE DISTILLATION STARTS HERE
KD: hard=0.50, soft=0.50, T=2.5
epoch 1/10 step 100/736 loss=9.0269 hard=2.4417 kd=15.6122 bb_lr=2.93e-06 elapsed=0.3m
epoch 1/10 step 200/736 loss=5.5364 hard=1.7425 kd=9.3304 bb_lr=5.81e-06 elapsed=0.6m
epoch 1/10 step 300/736 loss=4.0847 hard=1.3264 kd=6.8431 bb_lr=8.69e-06 elapsed=0.9m
epoch 1/10 step 400/736 loss=3.3137 hard=1.0941 kd=5.5332 bb_lr=1.16e-05 elapsed=1.2m
epoch 1/10 step 500/736 loss=2.7974 hard=0.9318 kd=4.6629 bb_lr=1.44e-05 elapsed=1.5m
epoch 1/10 step 600/736 loss=2.4328 hard=0.8189 kd=4.0468 bb_lr=1.69e-05 elapsed=1.8m
epoch 1/10 step 700/736 loss=2.1485 hard=0.7299 kd=3.5671 bb_lr=1.69e-05 elapsed=2.2m
epoch 1/10 step 736/736 loss=2.0598 hard=0.7002 kd=3.4194 bb_lr=1.69e-05 elapsed=2.3m
inference batch 50/374
inference batch 100/374
inference batch 150/374
inference batch 200/374
inference batch 250/374
inference batch 300/374
inference batch 350/374
   diagnostic: student predicted non-O=0.0862 (gold=0.0697)
Epoch 1:

In [20]:
def fresh_student():
    return Model2KDStudent(build_m2_backbone(),N_TAGS,HEAD_DROPOUT,HEAD_HIDDEN).to(DEVICE)

best_obj=torch.load(BEST_KD_PATH,map_location="cpu",weights_only=False)
student=fresh_student()
student.load_state_dict(best_obj["student_state"],strict=True)
BEST_THRESHOLD=best_obj["best_threshold"]
BEST_EPOCH=int(best_obj["best_epoch"])

val_pred=infer_dataset(student,val_ds,val_loader,BEST_THRESHOLD)
val_score=strict_scores(truth_val,val_pred,verbose=True)

with open(VAL_PRED_PATH,"w",encoding="utf-8") as f:
    json.dump(val_pred,f,ensure_ascii=False,indent=2)

(ROOT/"heldout_result.json").write_text(json.dumps({
    "experiment":"Phase2 Exp6 KD Model4-teacher -> Model2-student",
    "best_epoch":BEST_EPOCH,
    "threshold":BEST_THRESHOLD,
    "strict_macro_f1":float(val_score["macro_f1"]),
    "strict_macro_precision":float(val_score["macro_precision"]),
    "strict_macro_recall":float(val_score["macro_recall"]),
    "temperature":KD_TEMPERATURE,
    "hard_weight":HARD_LOSS_WEIGHT,
    "kd_weight":KD_LOSS_WEIGHT,
},indent=2),encoding="utf-8")

print("Best KD epoch:",BEST_EPOCH)
print("Best threshold(s):",BEST_THRESHOLD)
print("Saved:",VAL_PRED_PATH)


inference batch 50/374
inference batch 100/374
inference batch 150/374
inference batch 200/374
inference batch 250/374
inference batch 300/374
inference batch 350/374
APP       : P=0.292453 R=0.155000 F1=0.202614
RESP      : P=0.394958 R=0.247368 F1=0.304207
A.COUNSEL : P=0.593220 R=0.280000 F1=0.380435
R.COUNSEL : P=0.430769 R=0.577320 F1=0.493392
JUDGE     : P=0.598425 R=0.622951 F1=0.610442
WIT       : P=0.277778 R=0.041667 F1=0.072464
AUTH      : P=0.418675 R=0.297009 F1=0.347500
COURT     : P=0.866792 R=0.827957 F1=0.846929
STAT      : P=0.809375 R=0.704762 F1=0.753455
PREC      : P=0.510557 R=0.475850 F1=0.492593
DATE      : P=0.976776 R=0.938320 F1=0.957162
CASENO    : P=0.640288 R=0.528190 F1=0.578862
Strict Macro-F1:        0.503338
Strict Macro-Precision: 0.567505
Strict Macro-Recall:    0.474699
Best KD epoch: 7
Best threshold(s): {'APP': 0.1, 'RESP': 0.0, 'A.COUNSEL': 0.15, 'R.COUNSEL': 0.0, 'JUDGE': 0.0, 'WIT': 0.15, 'AUTH': 0.05, 'COURT': 0.65, 'STAT': 0.15, 'PREC': 0.1, 

## Final KD refit on all 70 labelled documents

Rebuild a clean Phase-1 Model-2 student and train for exactly `BEST_EPOCH` epochs on Fold1+Fold2. If `final_refit_partial_ner.pt` from Exp5 is attached, it is used as the stronger all-data teacher; otherwise the held-out Exp5 teacher is reused.


In [21]:
def build_teacher_from_task_checkpoint(task_path):
    obj=torch.load(task_path,map_location="cpu",weights_only=False)
    m=Model4ForNER(build_m4_backbone(),N_TAGS,HEAD_DROPOUT)
    # Trainable key structure is the same top-6 Exp5 structure.
    configure_teacher_structure(m)
    state=obj.get("trainable_state")
    assert state is not None
    load_named_trainable_state(m,state)
    for p in m.parameters(): p.requires_grad=False
    return m.to(DEVICE).eval(),obj

if not DO_FINAL_REFIT_ON_ALL:
    final_student=student
else:
    teacher_for_refit_path=TEACHER_REFIT_PATH if (TEACHER_REFIT_PATH and os.path.exists(TEACHER_REFIT_PATH)) else TEACHER_HELDOUT_PATH
    if teacher_for_refit_path!=TEACHER_HELDOUT_PATH:
        del teacher; gc.collect(); torch.cuda.empty_cache()    # free T4 memory before the 2nd teacher
        refit_teacher,refit_teacher_obj=build_teacher_from_task_checkpoint(teacher_for_refit_path)
    else:
        refit_teacher=teacher.eval()
    print("Final KD teacher:",teacher_for_refit_path)

    random.seed(SEED+1); np.random.seed(SEED+1); torch.manual_seed(SEED+1)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED+1)

    all_text=dict(fold1_text); all_text.update(fold2_text)
    all_gold=dict(fold1_gold); all_gold.update(fold2_gold)
    all_ds=NERWindowDataset(all_text,all_gold)
    all_loader=DataLoader(
        all_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True,
        collate_fn=collate_windows,num_workers=2,pin_memory=True
    )

    # all-data class weights
    FW=make_class_weights(all_ds)

    # Cache the final teacher once, then remove it from GPU before training the student.
    print("Caching final-refit teacher logits once...")
    REFIT_TEACHER_CACHE=precompute_teacher_logits(refit_teacher,all_ds)
    refit_teacher=refit_teacher.cpu()
    gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()

    final_student=fresh_student()
    opt=torch.optim.AdamW(make_student_param_groups(final_student),weight_decay=WEIGHT_DECAY)
    tp=[p for p in final_student.parameters() if p.requires_grad]
    upe=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)
    total=max(1,upe*BEST_EPOCH); warm=max(1,int(WARMUP_RATIO*total))
    def flr(step):
        if step<warm: return (step+1)/warm
        p=(step-warm)/max(1,total-warm)
        return 0.5*(1+math.cos(math.pi*min(1.0,p)))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,flr)

    for ep in range(1,BEST_EPOCH+1):
        final_student.train(); opt.zero_grad(set_to_none=True)
        rt=rh=rk=0.0; nbatches=0; t0=time.time()
        for step,batch in enumerate(all_loader,1):
            ids=batch["input_ids"].to(DEVICE,non_blocking=True)
            mask=batch["attention_mask"].to(DEVICE,non_blocking=True)
            y=batch["labels"].to(DEVICE,non_blocking=True)

            tl=teacher_logits_from_cache(batch,REFIT_TEACHER_CACHE,DEVICE)
            sl=final_student(ids,mask)

            hard=F.cross_entropy(
                sl.float().reshape(-1,N_TAGS),y.reshape(-1),
                weight=FW,ignore_index=IGNORE_INDEX
            )
            soft=kd_kl_loss(sl,tl,y,KD_TEMPERATURE,FW)
            loss=(HARD_LOSS_WEIGHT*hard+KD_LOSS_WEIGHT*soft)/GRAD_ACCUM_STEPS
            if not torch.isfinite(loss.detach()):
                raise FloatingPointError(f"Non-finite final KD loss ep={ep} batch={step}")

            loss.backward()
            rt+=float(loss.detach().cpu())*GRAD_ACCUM_STEPS
            rh+=float(hard.detach().cpu()); rk+=float(soft.detach().cpu()); nbatches+=1

            if step%GRAD_ACCUM_STEPS==0 or step==len(all_loader):
                gn=torch.nn.utils.clip_grad_norm_(tp,MAX_GRAD_NORM)
                if not torch.isfinite(torch.as_tensor(gn)):
                    raise FloatingPointError(f"Non-finite final KD grad ep={ep} batch={step}")
                opt.step(); opt.zero_grad(set_to_none=True); sched.step()

            if step%100==0 or step==len(all_loader):
                print(
                    f"refit epoch {ep}/{BEST_EPOCH} step {step}/{len(all_loader)} "
                    f"loss={rt/max(1,nbatches):.4f} hard={rh/max(1,nbatches):.4f} "
                    f"kd={rk/max(1,nbatches):.4f} elapsed={(time.time()-t0)/60:.1f}m"
                )

        # quick all-data training diagnostic
        with torch.no_grad():
            probe_batch=next(iter(DataLoader(all_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate_windows)))
            pids=probe_batch["input_ids"].to(DEVICE); pmask=probe_batch["attention_mask"].to(DEVICE)
            plg=final_student(pids,pmask)
            valid=pmask.bool()
            frac=float((plg.argmax(-1)[valid]!=O_ID).float().mean().cpu()) if bool(valid.any()) else 0.0
            print(f"   refit diagnostic: probe predicted non-O={frac:.4f}")

    torch.save({
        "student_state":student_state_dict(final_student),
        "best_epoch":BEST_EPOCH,
        "threshold":BEST_THRESHOLD,
        "teacher_checkpoint":teacher_for_refit_path,
        "kd_temperature":KD_TEMPERATURE,
        "hard_loss_weight":HARD_LOSS_WEIGHT,
        "kd_loss_weight":KD_LOSS_WEIGHT,
        "experiment":"Phase2 Exp6 KD final all-data refit",
        "tokenizer_sha256":sha256_file(TOKENIZER_PATH)
    },FINAL_REFIT_PATH)
    print("✅ Final KD student saved:",FINAL_REFIT_PATH)


Final KD teacher: /kaggle/input/datasets/aaryant23/phase2-kd/final_refit_partial_ner.pt
Built 1,484 windows from 70 docs in 0.0 min
Caching final-refit teacher logits once...
teacher cache 50/742
teacher cache 100/742
teacher cache 150/742
teacher cache 200/742
teacher cache 250/742
teacher cache 300/742
teacher cache 350/742
teacher cache 400/742
teacher cache 450/742
teacher cache 500/742
teacher cache 550/742
teacher cache 600/742
teacher cache 650/742
teacher cache 700/742
teacher cache 742/742
✅ Teacher logits cached: 1484 windows, 104.6 MiB, 1.6 min
refit epoch 1/7 step 100/1484 loss=9.4526 hard=2.5782 kd=16.3270 elapsed=0.3m
refit epoch 1/7 step 200/1484 loss=5.8701 hard=1.8208 kd=9.9194 elapsed=0.6m
refit epoch 1/7 step 300/1484 loss=4.4350 hard=1.4902 kd=7.3799 elapsed=0.9m
refit epoch 1/7 step 400/1484 loss=3.5804 hard=1.2366 kd=5.9242 elapsed=1.2m
refit epoch 1/7 step 500/1484 loss=2.9963 hard=1.0482 kd=4.9445 elapsed=1.5m
refit epoch 1/7 step 600/1484 loss=2.5792 hard=0.909

In [22]:
test_ds=NERWindowDataset(test_text,gold_map=None)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate_windows,num_workers=2,pin_memory=True)

def validate_prediction(pred,text_map,labels):
    if not isinstance(pred,dict) or not pred: raise ValueError("prediction must be non-empty dict")
    if set(pred)!=set(text_map): raise ValueError(f"ID mismatch missing={len(set(text_map)-set(pred))} extra={len(set(pred)-set(text_map))}")
    allowed=set(labels); n=0
    for did,ents in pred.items():
        if not isinstance(ents,list): raise ValueError(f"{did}: entities must be list")
        seen=set(); text=text_map[did]
        for j,s in enumerate(ents):
            if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f"{did} entity {j}: bad format")
            a,b,l=s[:3]
            if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError("offsets must be ints")
            if not (0<=a<b<=len(text)): raise ValueError(f"{did}: invalid span {a,b}")
            if l not in allowed: raise ValueError(f"{did}: invalid label {l}")
            t=(a,b,l)
            if t in seen: raise ValueError(f"{did}: duplicate {t}")
            seen.add(t); n+=1
    return n

def write_submission(pred,json_path,zip_path):
    n=validate_prediction(pred,test_text,LABELS)
    with open(json_path,"w",encoding="utf-8") as f: json.dump(pred,f,ensure_ascii=False)
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf: zf.write(json_path,arcname="prediction.json")
    with zipfile.ZipFile(zip_path,"r") as zf:
        names=zf.namelist(); assert names==["prediction.json"]
        validate_prediction(json.loads(zf.read("prediction.json").decode("utf-8")),test_text,LABELS)
    print(f"✅ {zip_path.name}: docs={len(pred)} spans={n} zip={names}")
    return n

variants={"heldout":student}
if final_student is not student: variants["refit"]=final_student
primary=variants.get("refit",student)
for name,m in variants.items():
    pred=infer_dataset(m,test_ds,test_loader,BEST_THRESHOLD)
    write_submission(pred,ROOT/f"prediction_{name}.json",ROOT/f"submission_{name}.zip")
    if m is primary:
        submission_pred=pred; n_spans=write_submission(pred,TEST_PRED_PATH,SUBMISSION_ZIP)
print("Upload BOTH submission_heldout.zip and submission_refit.zip (if present) to Codabench; keep the better one.")
print("✅ PHASE-2 SUBMISSION(S) PASSED ALL STRUCTURAL CHECKS")


Built 789 windows from 35 docs in 0.0 min
inference batch 50/395
inference batch 100/395
inference batch 150/395
inference batch 200/395
inference batch 250/395
inference batch 300/395
inference batch 350/395
✅ submission_heldout.zip: docs=35 spans=3656 zip=['prediction.json']
inference batch 50/395
inference batch 100/395
inference batch 150/395
inference batch 200/395
inference batch 250/395
inference batch 300/395
inference batch 350/395
✅ submission_refit.zip: docs=35 spans=3702 zip=['prediction.json']
✅ submission.zip: docs=35 spans=3702 zip=['prediction.json']
Upload BOTH submission_heldout.zip and submission_refit.zip (if present) to Codabench; keep the better one.
✅ PHASE-2 SUBMISSION(S) PASSED ALL STRUCTURAL CHECKS


In [24]:
EVALUATOR='#!/usr/bin/env python3\n"""Local evaluator for Phase 2.\n\nUsage:\n  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt\n"""\nimport argparse, json, sys\n\nclass DuplicateJSONKeyError(ValueError): pass\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\ndef pairs_no_duplicates(pairs):\n    out={}; dup=[]\n    for k,v in pairs:\n        if k in out: dup.append(k)\n        out[k]=v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\ndef load_json(path):\n    with open(path,encoding=\'utf-8\') as f:\n        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\ndef labels(path):\n    with open(path,encoding=\'utf-8\') as f: xs=[x.strip() for x in f if x.strip()]\n    if not xs: raise ValueError(\'labels.txt is empty.\')\n    if len(xs)!=12: raise ValueError(f\'Expected 12 labels, found {len(xs)}.\')\n    if len(set(xs))!=len(xs): raise ValueError(\'labels.txt contains duplicate labels.\')\n    return xs\ndef spans(x, allowed, where):\n    if x is None: return set()\n    if not isinstance(x,list): raise ValueError(f\'{where}: expected a list of entities.\')\n    out=set()\n    for j,s in enumerate(x):\n        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f\'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].\')\n        a,b,l=s[:3]\n        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f\'{where}, entity {j}: start/end must be integers.\')\n        if a<0 or b<=a: raise ValueError(f\'{where}, entity {j}: require 0 <= start < end.\')\n        if not isinstance(l,str) or l not in allowed: raise ValueError(f\'{where}, entity {j}: invalid label {l!r}.\')\n        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f\'{where}, entity {j}: entity_text must be a string.\')\n        t=(a,b,l)\n        if t in out: raise ValueError(f\'{where}, entity {j}: duplicate span {t}.\')\n        out.add(t)\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 2 entity predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); ap.add_argument(\'--labels\',required=True)\n    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)\n    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError(\'Truth and prediction must be non-empty JSON objects.\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    stats={l:[0,0,0] for l in labs}; allowed=set(labs)\n    for did in truth:\n        g=spans(truth[did],allowed,f\'truth[{did}]\'); p=spans(pred[did],allowed,f\'prediction[{did}]\')\n        for l in labs:\n            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}\n            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)\n    precisions=[]; recalls=[]; f1s=[]\n    print(f\'Examples: {len(truth)}\')\n    for l in labs:\n        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0\n        precisions.append(pr); recalls.append(rc); f1s.append(f)\n        print(f\'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}\')\n    print(f\'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}\')\n    print(f\'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}\')\n    print(f\'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
eval_path=ROOT/"evaluate_phase2.py"
eval_path.write_text(EVALUATOR,encoding="utf-8")
print("Evaluator written:",eval_path)


Evaluator written: /kaggle/working/phase2_exp6_kd_m4teacher_m2student/evaluate_phase2.py


In [25]:
print("KEEP THESE FILES:")
print(" Teacher Phase1 Model4:",MODEL4_PHASE1_PATH)
print(" Teacher Exp5 heldout:",TEACHER_HELDOUT_PATH)
if TEACHER_REFIT_PATH: print(" Teacher Exp5 refit:",TEACHER_REFIT_PATH)
print(" Student Phase1 Model2:",MODEL2_PHASE1_PATH)
print(" Exact tokenizer:",TOKENIZER_PATH)
print(" Best heldout KD student:",BEST_KD_PATH)
if FINAL_REFIT_PATH.exists(): print(" Final all-data KD student:",FINAL_REFIT_PATH)
if (ROOT/"heldout_result.json").exists(): print(" Heldout result:",ROOT/"heldout_result.json")
if SUBMISSION_ZIP.exists(): print(" submission.zip:",SUBMISSION_ZIP)

print("\nExperiment identity: KD — Exp5 Model4 teacher -> Phase1 Model2 student")
print("T:",KD_TEMPERATURE,"hard:",HARD_LOSS_WEIGHT,"KD:",KD_LOSS_WEIGHT)
print("Student head:",HEAD_TYPE,HEAD_HIDDEN)
print("Student top-block/head LR:",STUDENT_BACKBONE_LR,STUDENT_HEAD_LR)
print("Layerwise decay:",STUDENT_LAYERWISE_LR_DECAY)
print("KD cache/class balance:",CACHE_TEACHER_LOGITS,KD_WEIGHT_BY_CLASS)
if "BEST_EPOCH" in globals(): print("Best KD epoch:",BEST_EPOCH)
if "BEST_THRESHOLD" in globals(): print("Selected thresholds:",BEST_THRESHOLD)


KEEP THESE FILES:
 Teacher Phase1 Model4: /kaggle/input/datasets/aaryant23/phase2-kd/eff_full_best (6).pt
 Teacher Exp5 heldout: /kaggle/input/datasets/aaryant23/phase2-kd/best_partial_ner.pt
 Teacher Exp5 refit: /kaggle/input/datasets/aaryant23/phase2-kd/final_refit_partial_ner.pt
 Student Phase1 Model2: /kaggle/input/datasets/aaryant23/phase2-kd/dec_full_best (8).pt
 Exact tokenizer: /kaggle/input/datasets/aaryant23/phase2-kd/tokenizer.json
 Best heldout KD student: /kaggle/working/phase2_exp6_kd_m4teacher_m2student/best_kd_student.pt
 Final all-data KD student: /kaggle/working/phase2_exp6_kd_m4teacher_m2student/final_refit_kd_student.pt
 Heldout result: /kaggle/working/phase2_exp6_kd_m4teacher_m2student/heldout_result.json
 submission.zip: /kaggle/working/phase2_exp6_kd_m4teacher_m2student/submission.zip

Experiment identity: KD — Exp5 Model4 teacher -> Phase1 Model2 student
T: 2.5 hard: 0.5 KD: 0.5
Student head: bigru 256
Student top-block/head LR: 5e-05 0.001
Layerwise decay: 0.92

## Run order — Experiment 6 Knowledge Distillation v3

Attach on Kaggle:
- `eff_full_best (6).pt` — Phase-1 Model 4
- `best_partial_ner.pt` — Exp5 held-out teacher
- `dec_full_best (8).pt` — Phase-1 Model 2 student
- `tokenizer.json` — verified shared tokenizer
- `final_refit_partial_ner.pt` — recommended for final all-70-doc KD refit

Run top-to-bottom. **Before student training**, verify:
1. both Phase-1 checkpoints load strictly;
2. teacher/student output shapes match `[B,T,N_TAGS]`;
3. Fold-2 oracle is ~0.9987;
4. reconstructed teacher F1 matches the F1 stored inside `best_partial_ner.pt` within `5e-4`;
5. teacher logits are cached successfully and the teacher is moved off GPU.

Training starts at `🔥 ACTUAL KNOWLEDGE DISTILLATION STARTS HERE`.
